In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)
rng = np.random.default_rng(42)

def calcular_metricas_gap(real_vals, pred_vals):
    """Calcula RMSE y MAE entre valores reales y predichos."""
    valid = ~np.isnan(pred_vals) & ~np.isnan(real_vals)
    if valid.sum() == 0:
        return np.nan, np.nan
    diff = real_vals[valid] - pred_vals[valid]
    rmse = np.sqrt(np.mean(diff**2))
    mae = np.mean(np.abs(diff))
    return rmse, mae
from IPython.display import display, Markdown

# Semana 11B — Imputación de Datos Faltantes: Métodos Deterministas (Laboratorio)

**Objetivo:** Evaluar cuantitativamente ffill, interpolación lineal,
spline y día análogo sobre `tdb` de ClimaLab usando gaps artificiales
con ground truth.

**Ejercicios:**

1. Crear gaps artificiales en `tdb`
2. Aplicar los 4 métodos de imputación
3. Evaluación cuantitativa (RMSE, MAE)
4. Visualización del gap más largo
5. Conclusión: ¿qué método para qué tipo de gap?

In [ ]:
df = pd.read_parquet("data/ClimaLab_2023-05-31_2025-06-20.parquet")

In [ ]:
tdb_horaria = df["tdb"].resample("1h").mean().dropna()

---
## 1. Crear gaps artificiales en `tdb` horaria

Como `tdb` está completa, creamos gaps artificiales de 3 tipos
y guardamos los valores reales como ground truth para evaluar.

| Tipo | Duración | Cantidad |
|:---|:---|---:|
| Puntual | 1–3 horas | 10 |
| Mediano | 12–24 horas | 5 |
| Largo | 48–72 horas | 3 |

In [ ]:
tdb_con_gaps = tdb_horaria.copy()
ground_truth = {}
gap_info = []

# Elegir posiciones de inicio seguras (lejos de bordes y entre sí)
n_total = len(tdb_horaria)
margen = 200  # horas de margen al inicio y final

# Gaps puntuales: 1–3 horas
starts_punt = rng.choice(range(margen, n_total // 3), size=10, replace=False)
starts_punt.sort()
idx = 0
while idx < len(starts_punt):
    s = starts_punt[idx]
    duracion = int(rng.integers(1, 4))
    e = s + duracion
    gt_vals = tdb_con_gaps.iloc[s:e].copy()
    ground_truth[f"puntual_{idx}"] = gt_vals
    gap_info.append({"tipo": "puntual", "inicio": s, "fin": e, "horas": duracion})
    tdb_con_gaps.iloc[s:e] = np.nan
    idx += 1

# Gaps medianos: 12–24 horas
starts_med = rng.choice(range(n_total // 3, 2 * n_total // 3), size=5, replace=False)
starts_med.sort()
idx = 0
while idx < len(starts_med):
    s = starts_med[idx]
    duracion = int(rng.integers(12, 25))
    e = s + duracion
    gt_vals = tdb_con_gaps.iloc[s:e].copy()
    ground_truth[f"mediano_{idx}"] = gt_vals
    gap_info.append({"tipo": "mediano", "inicio": s, "fin": e, "horas": duracion})
    tdb_con_gaps.iloc[s:e] = np.nan
    idx += 1

# Gaps largos: 48–72 horas
starts_largo = rng.choice(range(2 * n_total // 3, n_total - margen), size=3, replace=False)
starts_largo.sort()
idx = 0
while idx < len(starts_largo):
    s = starts_largo[idx]
    duracion = int(rng.integers(48, 73))
    e = min(s + duracion, n_total)
    gt_vals = tdb_con_gaps.iloc[s:e].copy()
    ground_truth[f"largo_{idx}"] = gt_vals
    gap_info.append({"tipo": "largo", "inicio": s, "fin": e, "horas": e - s})
    tdb_con_gaps.iloc[s:e] = np.nan
    idx += 1

gap_info_df = pd.DataFrame(gap_info)

In [ ]:
resumen = gap_info_df.groupby("tipo")["horas"].agg(["count", "sum", "mean"]).round(1)
display(Markdown(f"""
### Gaps creados

| Tipo | Cantidad | Horas totales | Horas promedio |
|:---|---:|---:|---:|
| puntual | {resumen.loc["puntual", "count"]:.0f} | {resumen.loc["puntual", "sum"]:.0f} | {resumen.loc["puntual", "mean"]:.1f} |
| mediano | {resumen.loc["mediano", "count"]:.0f} | {resumen.loc["mediano", "sum"]:.0f} | {resumen.loc["mediano", "mean"]:.1f} |
| largo | {resumen.loc["largo", "count"]:.0f} | {resumen.loc["largo", "sum"]:.0f} | {resumen.loc["largo", "mean"]:.1f} |

NaN totales inyectados: **{resumen["sum"].sum():.0f}** horas
"""))

---
## 2. Aplicar los 4 métodos de imputación

In [ ]:
# 1. Forward fill
imp_ffill = tdb_con_gaps.ffill()

# 2. Interpolación lineal
imp_lineal = tdb_con_gaps.interpolate(method="linear")

# 3. Spline cúbica
imp_spline = tdb_con_gaps.interpolate(method="spline", order=3)

# 4. Día análogo: promediar misma hora en ±5 días
imp_analogo = tdb_con_gaps.copy()
nan_mask = tdb_con_gaps.isna()
nan_positions = np.where(nan_mask.values)[0]

pos_idx = 0
while pos_idx < len(nan_positions):
    pos = nan_positions[pos_idx]
    valores_ref = []
    dia_offset = 1
    while dia_offset <= 7:
        idx_antes = pos - dia_offset * 24
        idx_despues = pos + dia_offset * 24
        if 0 <= idx_antes < len(tdb_con_gaps):
            val = tdb_con_gaps.iloc[idx_antes]
            if not np.isnan(val):
                valores_ref.append(val)
        if 0 <= idx_despues < len(tdb_con_gaps):
            val = tdb_con_gaps.iloc[idx_despues]
            if not np.isnan(val):
                valores_ref.append(val)
        dia_offset += 1
    if len(valores_ref) > 0:
        imp_analogo.iloc[pos] = np.mean(valores_ref)
    pos_idx += 1

imputaciones = {
    "Forward fill": imp_ffill,
    "Lineal": imp_lineal,
    "Spline": imp_spline,
    "Día análogo": imp_analogo,
}

---
## 3. Evaluación cuantitativa (RMSE y MAE)

Calculamos RMSE y MAE por método y tipo de gap, usando el
ground truth (valores reales antes de inyectar NaN).

In [ ]:
display(metricas_df.round(3))

In [ ]:
plot_tipos = ["puntual", "mediano", "largo"]
plot_metodos = metricas_df["Método"].unique()
n_plot_metodos = len(plot_metodos)
x_pos = np.arange(len(plot_tipos))
bar_width = 0.18

fig3, (ax3a, ax3b) = plt.subplots(1, 2, figsize=(13, 5))

bar_colores = ["#3498db", "#e74c3c", "#2ecc71", "#e67e22"]

bar_idx = 0
while bar_idx < n_plot_metodos:
    met_name = plot_metodos[bar_idx]
    datos_met = metricas_df[metricas_df["Método"] == met_name]
    rmse_vals = []
    mae_vals = []
    t_idx = 0
    while t_idx < len(plot_tipos):
        tp = plot_tipos[t_idx]
        fila = datos_met[datos_met["Tipo de gap"] == tp]
        rmse_vals.append(fila["RMSE (°C)"].values[0] if len(fila) > 0 else 0)
        mae_vals.append(fila["MAE (°C)"].values[0] if len(fila) > 0 else 0)
        t_idx += 1

    offset = (bar_idx - n_plot_metodos / 2 + 0.5) * bar_width
    ax3a.bar(x_pos + offset, rmse_vals, bar_width, color=bar_colores[bar_idx],
             label=met_name, edgecolor="white")
    ax3b.bar(x_pos + offset, mae_vals, bar_width, color=bar_colores[bar_idx],
             label=met_name, edgecolor="white")
    bar_idx += 1

ax3a.set_xticks(x_pos)
ax3a.set_xticklabels(plot_tipos)
ax3a.set_ylabel("RMSE (°C)")
ax3a.set_title("RMSE por método y tipo de gap")
ax3a.legend(fontsize=9)

ax3b.set_xticks(x_pos)
ax3b.set_xticklabels(plot_tipos)
ax3b.set_ylabel("MAE (°C)")
ax3b.set_title("MAE por método y tipo de gap")
ax3b.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Visualización del gap más largo

Para el gap largo más extenso, graficamos las 4 imputaciones
superpuestas sobre el valor real.

In [ ]:
largos = gap_info_df[gap_info_df["tipo"] == "largo"].sort_values("horas", ascending=False)
gap_mayor = largos.iloc[0]
viz_start = int(gap_mayor["inicio"])
viz_end = int(gap_mayor["fin"])
viz_margen = 24  # horas de contexto

rango = slice(max(0, viz_start - viz_margen), min(len(tdb_horaria), viz_end + viz_margen))

fig4, ax4 = plt.subplots(figsize=(13, 5))
ax4.plot(tdb_horaria.index[rango], tdb_horaria.values[rango],
         "steelblue", lw=2, label="Real", zorder=5)

viz_colores = {"Forward fill": "#e74c3c", "Lineal": "#e67e22",
               "Spline": "#2ecc71", "Día análogo": "#9b59b6"}
viz_estilos = {"Forward fill": "--", "Lineal": "-.", "Spline": ":", "Día análogo": "-"}

viz_metodos = list(imputaciones.keys())
viz_i = 0
while viz_i < len(viz_metodos):
    nombre = viz_metodos[viz_i]
    serie = imputaciones[nombre]
    ax4.plot(serie.index[rango], serie.values[rango],
             color=viz_colores[nombre], ls=viz_estilos[nombre], lw=1.5,
             label=nombre, alpha=0.8)
    viz_i += 1

ax4.axvspan(tdb_horaria.index[viz_start], tdb_horaria.index[viz_end - 1],
            color="gray", alpha=0.1, label=f"Gap ({gap_mayor['horas']:.0f} h)")
ax4.set_xlabel("Fecha")
ax4.set_ylabel("Temperatura (°C)")
ax4.set_title(f"Gap largo ({gap_mayor['horas']:.0f} horas) — comparación de métodos")
ax4.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Conclusiones

> **Gaps puntuales (1–3 h):** Todos los métodos funcionan
> razonablemente.  Forward fill es el más simple y suficiente.
>
> **Gaps medianos (12–24 h):** Interpolación lineal y spline mejoran
> sobre ffill. Día análogo captura el ciclo diario.
>
> **Gaps largos (48–72 h):** Ffill y lineal fallan visiblemente.
> La spline puede oscilar. Día análogo es el mejor de los 4 métodos
> pero aún introduce ruido de los días de referencia.
>
> **Para gaps largos con estructura temporal, necesitamos un modelo
> que capture la dinámica de la serie → ARIMA/SARIMA (semanas 12–13).**

---

## Resumen de la sesión

| Ejercicio | Hallazgo principal |
|:---|:---|
| **Gaps artificiales** | 18 gaps (10 puntuales + 5 medianos + 3 largos) con ground truth |
| **4 métodos** | Ffill, lineal, spline, día análogo aplicados a los mismos gaps |
| **RMSE/MAE** | Ffill ~ lineal para puntuales; día análogo gana para largos |
| **Visualización** | Ffill se queda plano; lineal ignora ciclos; spline oscila; día análogo reproduce el patrón diario |

**Próxima semana (12):** Modelos ARIMA — el primer modelo que
*entiende* la dinámica temporal.